In [1]:
!pip install numpy qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 68.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 86.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 3.9 MB/s eta 0:00:00


In [2]:
!pip install git+https://github.com/stephenhky/QAlgo.git@develop

  Cloning https://github.com/stephenhky/QAlgo.git (to revision develop) to /tmp/pip-req-build-9x5g0dlr
  Running command git clone --filter=blob:none --quiet https://github.com/stephenhky/QAlgo.git /tmp/pip-req-build-9x5g0dlr
  Running command git checkout -b develop --track origin/develop
  Switched to a new branch 'develop'
  Branch 'develop' set up to track remote branch 'develop' from 'origin'.
  Resolved https://github.com/stephenhky/QAlgo.git to commit bd5cb4df09c6cbef25d05315b6ddb28871a61538
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for qalgo: filename=qalgo-0.0.2a1-py3-none-any.whl size=4696 sha256=b8dce915e92f6f7a841ecd29723d96ae69e190932bf6cef76058946a292ca791
  Stored in directory: /tmp/pip-ephem-wheel-cache-n7zhorkm/wheels/18/07/e8/d052d744e1e52c4cbba496a94efc1f79bda87751160b756285
Successfully built qalgo


In [3]:
from typing import Annotated

import numpy as np
import numpy.typing as npt
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister, Gate
from qiskit import transpile
from qiskit.circuit.library import StatePreparation, HamiltonianGate
from qiskit.quantum_info import Statevector
from qiskit_aer import StatevectorSimulator

from qalgo.phase import PhaseEstimationGate, InversePhaseEstimationGate

In [16]:
sv_check = [None, None, None, None, None]

def HHLGate(
        A: Annotated[npt.NDArray[np.float64], "2D Hermitian matrix"],
        b: Annotated[npt.NDArray[np.float64], "1D array"],
        nb_b: int,   # b qubit
        nb_clock: int,   # clock qubit
        nb_ancilla: int = 1,    # ancilla qubit
        time: float | np.float64 = 2*np.pi/4
) -> tuple[Gate, Statevector]:
    assert nb_ancilla == 1     # ancilla must have only one qubit
    np.testing.assert_array_almost_equal(A, A.conj().T)   # test Hermitianity

    # initiating the circuit
    b_register = QuantumRegister(nb_b)
    clock_register = QuantumRegister(nb_clock)
    ancilla_register = QuantumRegister(nb_ancilla)
    qc = QuantumCircuit(ancilla_register, clock_register, b_register)
    sv_check[0] = Statevector(qc)

    # initiating b
    b_state_prep = StatePreparation(b)
    qc.append(b_state_prep, [b_register[i] for i in range(nb_b)])
    sv_check[1] = Statevector(qc)

    # initiating evolution gate
    evolution_gate = HamiltonianGate(A, time=time)
 
    # forward quantum phase estimation
    qc.append(
        PhaseEstimationGate(evolution_gate, nb_clock, nb_b),
        [clock_register[i] for i in range(nb_clock)] + [b_register[i] for i in range(nb_b)]
    )
    sv_check[2] = Statevector(qc)

    # controlled rotation
    for i in range(nb_clock):
        qc.cry(np.pi / (2**i), [clock_register[i] for i in range(nb_clock)], ancilla_register[0])
    sv_check[3] = Statevector(qc)

    # reverse quantum phase estimation
    qc.append(
        InversePhaseEstimationGate(evolution_gate.inverse(), nb_clock, nb_b),
        [clock_register[i] for i in range(nb_clock)] + [b_register[i] for i in range(nb_b)]
    )
    sv_check[4] = Statevector(qc)

    return qc.to_gate()

In [17]:
A = np.array([[1., -np.reciprocal(3.)], [-np.reciprocal(3.), 1.]])
b = np.array([0., 1.])

nb_b = 1
nb_clock = 4

ancilla_register = QuantumRegister(1)
clock_register = QuantumRegister(nb_clock)
b_register = QuantumRegister(nb_b)
# classical_register = ClassicalRegister(1)
qc = QuantumCircuit(ancilla_register, clock_register, b_register)
sv0 = Statevector(qc)
qc.append(
    HHLGate(A, b, nb_b, nb_clock),
    [ancilla_register[0]] + [clock_register[i] for i in range(nb_clock)] + [b_register[i] for i in range(nb_b)]
)

sv1 = Statevector(qc)

In [25]:
sv_check[0].draw("latex")

<IPython.core.display.Latex object>

In [26]:
sv_check[1].draw("latex")

<IPython.core.display.Latex object>

In [27]:
sv_check[2].draw("latex")

<IPython.core.display.Latex object>

In [28]:
sv_check[3].draw("latex")

<IPython.core.display.Latex object>

In [29]:
sv_check[4].draw("latex")

<IPython.core.display.Latex object>

In [12]:
sv0.draw("latex")

<IPython.core.display.Latex object>

In [13]:
sv1.draw("latex")

<IPython.core.display.Latex object>

In [23]:
filtered_probs = {
    str(key): float(prob)
    for key, prob in sv1.probabilities_dict().items()
    if key[-1] == '1'
}

In [24]:
{
    "0": sum(val for key, val in filtered_probs.items() if key[0]=="0"),
    "1": sum(val for key, val in filtered_probs.items() if key[0]=="1")
}

{'0': 0.1296869286407786, '1': 0.14832809982527415}